# Projeto de Estatística — futebol e mercado de transferências

Este notebook marca o início da análise do dataset **Football Data from Transfermarkt**, disponibilizado no Kaggle por [David Cariboo](https://www.kaggle.com/datasets/davidcariboo/player-scores). O conjunto reúne partidas, clubes, resultados por clube e transferências de jogadores.

A primeira etapa é carregar e validar as tabelas. A partir daí, podemos investigar perguntas como:

- Quais clubes apresentam melhor desempenho por temporada?
- Existe relação entre investimento líquido em transferências e vitórias?
- Como variam as taxas de transferência e os valores de mercado ao longo do tempo?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Permite executar o notebook tanto a partir da raiz do projeto quanto de uma pasta de notebooks.
DATA_DIR = Path("dados")
if not DATA_DIR.exists():
    DATA_DIR = Path("../dados")

clubs = pd.read_csv(DATA_DIR / "clubs.csv")
games = pd.read_csv(DATA_DIR / "games.csv", parse_dates=["date"])
club_games = pd.read_csv(DATA_DIR / "club_games.csv")
transfers = pd.read_csv(DATA_DIR / "transfers.csv", parse_dates=["transfer_date"])

print(f"Diretório de dados: {DATA_DIR.resolve()}")
print("Tabelas carregadas.")

In [ ]:
tamanhos = pd.DataFrame({
    "tabela": ["clubs", "games", "club_games", "transfers"],
    "linhas": [len(clubs), len(games), len(club_games), len(transfers)],
    "colunas": [clubs.shape[1], games.shape[1], club_games.shape[1], transfers.shape[1]],
})
tamanhos

## Qualidade inicial dos dados

A tabela abaixo ajuda a identificar campos incompletos antes de escolher as variáveis para a análise inferencial. Valores ausentes são esperados em campos como presença, árbitro ou treinador, dependendo da competição e da temporada.

In [ ]:
def resumo_nulos(df, nome):
    return (
        df.isna().sum()
        .rename("nulos")
        .to_frame()
        .assign(percentual=lambda x: (100 * x["nulos"] / len(df)).round(2), tabela=nome)
        .reset_index(names="coluna")
    )

qualidade = pd.concat(
    [resumo_nulos(clubs, "clubs"), resumo_nulos(games, "games"),
     resumo_nulos(club_games, "club_games"), resumo_nulos(transfers, "transfers")],
    ignore_index=True,
)
qualidade.query("nulos > 0").sort_values(["tabela", "nulos"], ascending=[True, False]).head(20)

## Desempenho dos clubes

`club_games` registra cada partida do ponto de vista de um clube. Assim, podemos calcular jogos, vitórias, empates, derrotas e aproveitamento sem duplicar manualmente as partidas.

In [ ]:
club_games = club_games.copy()
club_games["resultado"] = np.select(
    [club_games["is_win"] == 1, club_games["own_goals"] == club_games["opponent_goals"]],
    ["Vitória", "Empate"],
    default="Derrota",
)

desempenho = (
    club_games.groupby("club_id")
    .agg(
        jogos=("game_id", "nunique"),
        vitorias=("resultado", lambda s: (s == "Vitória").sum()),
        empates=("resultado", lambda s: (s == "Empate").sum()),
        gols_marcados=("own_goals", "sum"),
        gols_sofridos=("opponent_goals", "sum"),
    )
    .assign(aproveitamento=lambda x: (3 * x["vitorias"] + x["empates"]) / (3 * x["jogos"]))
    .reset_index()
)

desempenho = desempenho.merge(clubs[["club_id", "name"]], on="club_id", how="left")
desempenho.sort_values(["aproveitamento", "jogos"], ascending=False).head(10)

## Transferências

Os valores monetários vêm como texto no arquivo original. A conversão abaixo cria colunas numéricas em euros e permite resumir o volume financeiro por temporada. Transferências gratuitas aparecem como zero.

In [ ]:
transfers = transfers.copy()
for coluna in ["transfer_fee", "market_value_in_eur"]:
    transfers[coluna + "_eur"] = pd.to_numeric(transfers[coluna], errors="coerce")

transferencias_por_temporada = (
    transfers.groupby("transfer_season")
    .agg(
        transferencias=("player_id", "size"),
        taxa_total_eur=("transfer_fee_eur", "sum"),
        valor_medio_eur=("market_value_in_eur_eur", "mean"),
    )
    .sort_index()
)
transferencias_por_temporada.tail(10)

## Próximos passos

1. Definir a pergunta estatística e a população/amostra de interesse.
2. Escolher uma janela de temporadas e competições comparáveis.
3. Tratar valores ausentes e documentar os critérios de inclusão.
4. Explorar distribuições, correlações e possíveis modelos estatísticos.
5. Registrar conclusões, limitações e referências no próprio notebook.